# Lezione 2 — Programmazione quantistica con Cirq

In questa lezione passi dalla teoria al codice. Scriverai il tuo primo vero programma quantistico con Cirq, la libreria Python che Google ha pubblicato per progettare, simulare ed eseguire circuiti quantistici. È la base su cui si costruisce TensorFlow Quantum, quindi tutto ciò che vedi oggi lo riutilizzerai in ogni lezione successiva.

**Obiettivi di apprendimento**

- Costruire circuiti con `cirq.GridQubit` e `cirq.Circuit`
- Applicare porte a un qubit e a due qubit
- Usare simboli di `sympy` per creare circuiti addestrabili

In [ ]:
!pip install -q cirq

## I qubit vivono su una griglia

Parti da una domanda semplice: dove vive un qubit dentro Cirq? Non è un'idea astratta sospesa nel vuoto, ha una posizione precisa. La classe `cirq.GridQubit(riga, colonna)` rappresenta un qubit fisico collocato su una griglia, esattamente come sono distribuiti i qubit su un chip reale come Sycamore.

Pensa a una scacchiera: ogni pezzo occupa una casella precisa, con la sua riga e la sua colonna. Un `GridQubit` funziona allo stesso modo. Quando definisci `cirq.GridQubit(0, 0)` e `cirq.GridQubit(0, 1)`, stai posizionando due qubit uno accanto all'altro sulla stessa riga della griglia.

## Un circuito è una ricetta di passaggi

Un `cirq.Circuit()` è la sequenza ordinata di porte che applichi ai tuoi qubit. Pensalo come una ricetta di cucina: la ricetta non funziona se mescoli i passaggi, e neanche un circuito quantistico. Aggiungi ogni porta con `circuito.append(...)`, e Cirq le organizza automaticamente in "momenti": un momento raggruppa le porte che vengono eseguite contemporaneamente, su qubit diversi, per sfruttare il parallelismo dell'hardware.

Non devi gestire tu questo raggruppamento, Cirq lo fa per te. Il tuo compito è decidere quali porte applicare e in che ordine.

## Il catalogo delle porte che userai di più

Queste sono le porte che ripeterai praticamente in ogni circuito del corso:

- `cirq.H(qubit)`: Hadamard, crea sovrapposizione
- `cirq.X(qubit)`, `cirq.Y(qubit)`, `cirq.Z(qubit)`: rotazioni di 180 gradi attorno a ciascun asse, l'equivalente quantistico di "ribaltare" il qubit
- `cirq.rx(angolo)`, `cirq.ry(angolo)`, `cirq.rz(angolo)`: rotazioni di un angolo qualsiasi attorno a ciascun asse. Sono quelle che usi per "addestrare" un circuito, perché l'angolo può essere un parametro regolabile
- `cirq.CNOT(controllo, bersaglio)` e `cirq.CZ(controllo, bersaglio)`: porte a due qubit che creano entanglement
- Potenze frazionarie, come `cirq.X(qubit)**0.5`: applicano solo "una parte" della rotazione. `cirq.X(qubit)**i` equivale a una rotazione di `pi * i` radianti, un modo compatto di codificare un numero reale come angolo di rotazione (lo userai più avanti per convertire dati classici in stati quantistici)

## Parametri simbolici con sympy

Ecco il pezzo che trasforma un circuito quantistico in qualcosa di addestrabile. Cirq si appoggia alla libreria `sympy` per definire porte con un angolo che non conosci ancora: un simbolo al posto di un numero fisso.

Definisci la porta con un simbolo, per esempio `theta`, e più avanti un ottimizzatore prova diversi valori di `theta` fino a trovare quello che risolve meglio il tuo problema. Questa idea, "un parametro simbolico che viene regolato durante l'addestramento", è l'equivalente quantistico di un peso in una rete neurale classica. Tieni bene a mente questa analogia, la userai in ogni lezione da qui in avanti.

## Simulare e misurare

Senza accesso a un computer quantistico reale, simuli il circuito sul tuo computer con `cirq.Simulator()`. Il metodo `.run(circuito, repetitions=N)` esegue il circuito N volte, perché ogni singola esecuzione ti dà un solo risultato (ricorda la lezione 1), e restituisce la statistica dei risultati.

Vediamo tutto insieme nell'esempio completo.

In [ ]:
import cirq
import sympy

# Dos qubits en una rejilla de una sola fila
q0, q1 = cirq.GridQubit(0, 0), cirq.GridQubit(0, 1)

# Un símbolo: un ángulo que todavía no tiene valor numérico
theta = sympy.Symbol("theta")

circuito = cirq.Circuit()

# Ponemos el primer qubit en superposición
circuito.append(cirq.H(q0))

# Entrelazamos los dos qubits
circuito.append(cirq.CNOT(q0, q1))

# Rotamos el segundo qubit un ángulo "theta" sobre el eje Y (parámetro entrenable)
circuito.append(cirq.ry(theta).on(q1))

# Medimos los dos qubits
circuito.append(cirq.measure(q0, q1, key="resultado"))

print(circuito)

Osserva il diagramma che hai appena stampato: vedi le due linee dei qubit e, in ordine, la Hadamard, la CNOT e la rotazione con il simbolo `theta` ancora senza valore. Il circuito è completo, ma non puoi simularlo così com'è, perché `theta` non è un numero. Ti serve un `cirq.ParamResolver` che gli assegni un valore concreto.

In [ ]:
# Para simular, sustituimos el símbolo "theta" por un valor concreto
simulador = cirq.Simulator()
resolver = cirq.ParamResolver({"theta": 0.5})
resultado = simulador.run(circuito, resolver, repetitions=500)

print(resultado.histogram(key="resultado"))

Il `cirq.ParamResolver` è il meccanismo che sostituisce il simbolo `theta` con un numero concreto subito prima di simulare. Tieni bene a mente questo schema: circuito con simboli più un resolver che assegna i valori. È esattamente quello che TensorFlow Quantum usa sotto il cofano quando addestra un modello, solo che invece di assegnare il valore a mano, un ottimizzatore lo regola automaticamente, passo dopo passo, fino a minimizzare un errore.

## Riepilogo dei punti chiave

- `cirq.GridQubit` colloca un qubit in una posizione precisa di una griglia, come su un chip reale
- `cirq.Circuit()` e `.append(...)` costruiscono la sequenza di porte del circuito
- Le porte di rotazione (`rx`, `ry`, `rz`) accettano simboli di `sympy` come angolo, il che rende il circuito "addestrabile"
- Le potenze frazionarie di una porta (`cirq.X(q)**i`) codificano un numero reale come rotazione
- `cirq.Simulator()` esegue il circuito; `cirq.ParamResolver` sostituisce i simboli con valori concreti prima di simulare